# Implementasi Bab 8 — Support Vector Machine

Scaling, GridSearchCV, evaluasi, heatmap parameter, perbandingan kernel, dan decision boundary dua fitur [ML-MODUL-01, hlm. 58–59].


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=SEED, stratify=data.target
)
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC()),
])


In [ ]:
parameter_grid = {
    "svm__C": [0.1, 1, 10],
    "svm__gamma": [0.001, 0.01, 0.1],
    "svm__kernel": ["rbf", "linear"],
}
grid_search = GridSearchCV(
    svm_pipeline, parameter_grid, cv=5, scoring="f1", n_jobs=-1
)
grid_search.fit(X_train, y_train)
print("Parameter terbaik:", grid_search.best_params_)
print("F1 CV terbaik:", round(grid_search.best_score_, 3))


In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
decision_scores = best_model.decision_function(X_test)
print("Akurasi test:", round(accuracy_score(y_test, y_pred), 3))
print("Rentang decision_function:", decision_scores.min(), decision_scores.max())
print(classification_report(y_test, y_pred, target_names=data.target_names))


In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
rbf_results = cv_results[cv_results["param_svm__kernel"] == "rbf"]
pivot = rbf_results.pivot_table(
    values="mean_test_score", index="param_svm__C", columns="param_svm__gamma"
)
plt.imshow(pivot, cmap="viridis", aspect="auto")
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.xlabel("gamma")
plt.ylabel("C")
plt.title("Heatmap F1 CV — Kernel RBF")
plt.colorbar(label="F1")
plt.show()


## Latihan 1: perbandingan kernel

Semua kernel berada dalam Pipeline yang sama agar scaling dipelajari hanya dari train set [ML-MODUL-01, hlm. 59].


In [ ]:
kernel_cv_scores = {}
kernel_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for kernel in ["linear", "poly", "rbf"]:
    candidate = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel=kernel, C=1)),
    ])
    kernel_cv_scores[kernel] = cross_val_score(
        candidate, X_train, y_train, cv=kernel_cv, scoring="accuracy"
    ).mean()
print("Rata-rata akurasi CV per kernel:", kernel_cv_scores)


In [ ]:
selected_features = ["mean radius", "mean texture"]
X_two = data.data[selected_features].to_numpy()
y_two = data.target.to_numpy()
scaler_two = StandardScaler()
X_two_scaled = scaler_two.fit_transform(X_two)
x_min, x_max = X_two_scaled[:, 0].min() - 0.5, X_two_scaled[:, 0].max() + 0.5
y_min, y_max = X_two_scaled[:, 1].min() - 0.5, X_two_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
boundary_configs = [
    {"kernel": "linear", "C": 1},
    {"kernel": "rbf", "C": 1, "gamma": "scale"},
    {"kernel": "rbf", "C": 10, "gamma": 0.1},
]
fig, boundary_axes = plt.subplots(1, len(boundary_configs), figsize=(16, 4.5))
for axis, config in zip(boundary_axes, boundary_configs):
    boundary_model = SVC(**config).fit(X_two_scaled, y_two)
    zz = boundary_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    axis.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
    axis.scatter(X_two_scaled[:, 0], X_two_scaled[:, 1], c=y_two, s=12, cmap="coolwarm")
    axis.set_xlabel(selected_features[0] + " (scaled)")
    axis.set_ylabel(selected_features[1] + " (scaled)")
    axis.set_title(str(config))
fig.suptitle("Decision Boundary SVM — Perbandingan Kernel/C/Gamma")
plt.tight_layout()
plt.show()


## Kesimpulan

C mengatur toleransi pelanggaran margin, gamma mengatur jangkauan pengaruh sampel pada RBF, dan kernel menentukan bentuk batas keputusan. Keduanya harus dituning melalui CV pada training set [ML-MODUL-01, hlm. 59].
